# Hugging face dataset without NA/JNK

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from skmultilearn.model_selection import iterative_train_test_split

c:\Users\alrazz\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
INPUT_FILE = Path(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA\Combined_single_no_NA.jsonl"
)

# Automatically use the input filename as the dataset name
DATASET_NAME = INPUT_FILE.stem

OUTPUT_ROOT = Path(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA"
)

OUTPUT_DIR = OUTPUT_ROOT / DATASET_NAME

SEED = 44

TRAIN_SIZE = 0.70
DEV_SIZE = 0.15
TEST_SIZE = 0.15

In [19]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [20]:
labels_structure = {
    "MT": [],
    "LY": [],
    "SP": ["it"],
    "ID": [],
    "NA": ["ne", "sr", "nb"],
    "HI": ["re"],
    "IN": ["en", "ra", "dtp", "fi", "lt"],
    "OP": ["rv", "ob", "rs", "av"],
    "IP": ["ds", "ed"],
}

all_valid_labels = sorted(
    list(labels_structure.keys())
    + [s for subs in labels_structure.values() for s in subs]
)

print("Labels:")
print(all_valid_labels)
print(f"Number of labels: {len(all_valid_labels)}")


Labels:
['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'ra', 're', 'rs', 'rv', 'sr']
Number of labels: 25


In [21]:
# ============================================================
# Load JSONL
# ============================================================

def load_jsonl_data(filepath):
    texts = []
    labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            if "text" not in record:
                raise ValueError(
                    f"Missing 'text' field on line {line_number}"
                )

            if "label" not in record:
                raise ValueError(
                    f"Missing 'label' field on line {line_number}"
                )

            text = record["text"]
            label_list = record["label"].split()

            # Convert multilabel annotation into binary vector
            label_vector = [
                1.0 if label in label_list else 0.0
                for label in all_valid_labels
            ]

            texts.append(text)
            labels.append(label_vector)

    return (
        np.array(texts, dtype=object),
        np.array(labels, dtype=np.float32),
    )

In [22]:
# ============================================================
# Create Hugging Face Dataset
# ============================================================

def create_dataset(X, y):
    return Dataset.from_dict(
        {
            "text": X.tolist(),
            "labels": y.tolist(),
        }
    )


In [23]:
# ============================================================
# Load data
# ============================================================

print("\nLoading data...")

X, y = load_jsonl_data(INPUT_FILE)

print(f"Total examples: {len(X)}")
print(f"Label matrix shape: {y.shape}")



Loading data...
Total examples: 4205
Label matrix shape: (4205, 25)


In [24]:
# ============================================================
# Validate split sizes
# ============================================================

assert abs(TRAIN_SIZE + DEV_SIZE + TEST_SIZE - 1.0) < 1e-8


In [25]:
# ============================================================
# 1. Train vs temporary (dev + test)
# ============================================================

print("\nCreating train/temp split...")

X_2d = X.reshape(-1, 1)

X_train, y_train, X_temp, y_temp = iterative_train_test_split(
    X_2d,
    y,
    test_size=(DEV_SIZE + TEST_SIZE),
)



Creating train/temp split...


In [26]:
# ============================================================
# 2. Dev vs test
# ============================================================

print("Creating dev/test split...")

# Since dev and test should each be 15% of the full dataset,
# they are 50/50 within the temporary 30% subset.

X_dev, y_dev, X_test, y_test = iterative_train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
)


Creating dev/test split...


In [27]:
# ============================================================
# Flatten text arrays
# ============================================================

X_train = X_train.flatten()
X_dev = X_dev.flatten()
X_test = X_test.flatten()


In [28]:
# ============================================================
# Create Hugging Face datasets
# ============================================================

print("\nCreating Hugging Face datasets...")

train_dataset = create_dataset(X_train, y_train)
dev_dataset = create_dataset(X_dev, y_dev)
test_dataset = create_dataset(X_test, y_test)


Creating Hugging Face datasets...


In [29]:
# ============================================================
# Deterministic shuffle
# ============================================================

train_dataset = train_dataset.shuffle(seed=SEED)
dev_dataset = dev_dataset.shuffle(seed=SEED)
test_dataset = test_dataset.shuffle(seed=SEED)

In [30]:
# ============================================================
# Create DatasetDict
# ============================================================

dataset_dict = DatasetDict(
    {
        "train": train_dataset,
        "dev": dev_dataset,
        "test": test_dataset,
    }
)



In [31]:
# ============================================================
# Print split information
# ============================================================

print("\nFinal split:")
print(f"Train: {len(dataset_dict['train'])}")
print(f"Dev:   {len(dataset_dict['dev'])}")
print(f"Test:  {len(dataset_dict['test'])}")
print(f"Total: {sum(len(dataset_dict[x]) for x in ['train', 'dev', 'test'])}")


Final split:
Train: 2963
Dev:   609
Test:  633
Total: 4205


In [32]:
# ============================================================
# Check label distributions
# ============================================================

print("\nLabel distributions:")

for split_name in ["train", "dev", "test"]:

    split_labels = np.array(
        dataset_dict[split_name]["labels"],
        dtype=np.float32,
    )

    counts = split_labels.sum(axis=0)

    print(f"\n{split_name}:")
    for label, count in zip(all_valid_labels, counts):
        print(f"  {label:5s}: {int(count)}")



Label distributions:

train:
  HI   : 288
  ID   : 273
  IN   : 993
  IP   : 514
  LY   : 220
  MT   : 146
  NA   : 818
  OP   : 645
  SP   : 284
  av   : 188
  ds   : 233
  dtp  : 334
  ed   : 118
  en   : 106
  fi   : 125
  it   : 150
  lt   : 185
  nb   : 131
  ne   : 367
  ob   : 168
  ra   : 120
  re   : 125
  rs   : 130
  rv   : 66
  sr   : 162

dev:
  HI   : 61
  ID   : 59
  IN   : 212
  IP   : 110
  LY   : 47
  MT   : 31
  NA   : 175
  OP   : 138
  SP   : 60
  av   : 40
  ds   : 50
  dtp  : 72
  ed   : 25
  en   : 22
  fi   : 27
  it   : 32
  lt   : 39
  nb   : 28
  ne   : 79
  ob   : 36
  ra   : 26
  re   : 26
  rs   : 28
  rv   : 14
  sr   : 35

test:
  HI   : 62
  ID   : 58
  IN   : 213
  IP   : 111
  LY   : 48
  MT   : 31
  NA   : 176
  OP   : 139
  SP   : 61
  av   : 41
  ds   : 50
  dtp  : 71
  ed   : 25
  en   : 23
  fi   : 27
  it   : 32
  lt   : 40
  nb   : 28
  ne   : 79
  ob   : 36
  ra   : 26
  re   : 27
  rs   : 28
  rv   : 15
  sr   : 35


In [33]:
# ============================================================
# Save DatasetDict
# ============================================================

print(f"\nSaving dataset to: {OUTPUT_DIR}")

# Prevent accidentally overwriting an existing split
if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"\nOutput directory already exists:\n"
        f"  {OUTPUT_DIR}\n\n"
        f"Delete it manually if you intentionally want to recreate "
        f"the fixed split."
    )

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

dataset_dict.save_to_disk(str(OUTPUT_DIR))



Saving dataset to: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_single_no_NA


Saving the dataset (1/1 shards): 100%|██████████| 633/633 [00:00<00:00, 62060.13 examples/s]


In [34]:
# ============================================================
# Save metadata
# ============================================================

metadata = {
    "dataset_name": DATASET_NAME,
    "input_file": str(INPUT_FILE),
    "seed": SEED,
    "train_size": TRAIN_SIZE,
    "dev_size": DEV_SIZE,
    "test_size": TEST_SIZE,
    "num_examples": len(X),
    "num_labels": len(all_valid_labels),
    "labels": all_valid_labels,
    "label_structure": labels_structure,
    "split_method": "skmultilearn.iterative_train_test_split",
    "shuffle_seed": SEED,
}

metadata_file = OUTPUT_DIR / "split_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


In [35]:
# ============================================================
# Final verification
# ============================================================

print("\nVerifying saved dataset...")

loaded_dataset = DatasetDict.load_from_disk(str(OUTPUT_DIR))

assert len(loaded_dataset["train"]) == len(dataset_dict["train"])
assert len(loaded_dataset["dev"]) == len(dataset_dict["dev"])
assert len(loaded_dataset["test"]) == len(dataset_dict["test"])

print("Verification successful.")

print("\nSaved:")
print(f"  {OUTPUT_DIR}")
print(f"  {metadata_file}")

print("\nYou can now load the fixed split with:")
print()
print('  from datasets import load_from_disk')
print()
print(
    f'  dataset = load_from_disk("{OUTPUT_DIR}")'
)
print()
print('  train_dataset = dataset["train"]')
print('  dev_dataset   = dataset["dev"]')
print('  test_dataset  = dataset["test"]')


Verifying saved dataset...
Verification successful.

Saved:
  C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_single_no_NA
  C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_single_no_NA\split_metadata.json

You can now load the fixed split with:

  from datasets import load_from_disk

  dataset = load_from_disk("C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_single_no_NA")

  train_dataset = dataset["train"]
  dev_dataset   = dataset["dev"]
  test_dataset  = dataset["test"]


# Create matched Hugging Face datasets using the fixed reference split
#
The split from Combined_single_no_NA.jsonl is treated as the reference split. All other JSONL files must contain exactly the same instances and will use the exact same train/dev/test assignment, but with their own labels.

In [36]:
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
from datasets import Dataset, DatasetDict


# ============================================================
# Configuration
# ============================================================

INPUT_FOLDER = Path(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA"
)

REFERENCE_FILE = INPUT_FOLDER / "Combined_single_no_NA.jsonl"

OUTPUT_ROOT = Path(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA"
)

SEED = 44

In [37]:
labels_structure = {
    "MT": [],
    "LY": [],
    "SP": ["it"],
    "ID": [],
    "NA": ["ne", "sr", "nb"],
    "HI": ["re"],
    "IN": ["en", "ra", "dtp", "fi", "lt"],
    "OP": ["rv", "ob", "rs", "av"],
    "IP": ["ds", "ed"],
}

all_valid_labels = sorted(
    list(labels_structure.keys())
    + [s for subs in labels_structure.values() for s in subs]
)

print(all_valid_labels)

['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'ra', 're', 'rs', 'rv', 'sr']


In [38]:
def load_jsonl(filepath):
    texts = []
    labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            if "text" not in record:
                raise ValueError(
                    f"Missing 'text' field in {filepath}, "
                    f"line {line_number}"
                )

            if "label" not in record:
                raise ValueError(
                    f"Missing 'label' field in {filepath}, "
                    f"line {line_number}"
                )

            texts.append(record["text"])
            labels.append(record["label"].split())

    return texts, labels

In [39]:
print("Loading reference file...")

reference_texts, reference_labels = load_jsonl(
    REFERENCE_FILE
)

print(
    f"Reference dataset contains "
    f"{len(reference_texts)} instances."
)

Loading reference file...
Reference dataset contains 4205 instances.


In [40]:
reference_dataset = DatasetDict.load_from_disk(
    str(OUTPUT_ROOT / REFERENCE_FILE.stem)
)

print("\nLoaded fixed reference dataset:")

print(
    f"Train: {len(reference_dataset['train'])}"
)

print(
    f"Dev:   {len(reference_dataset['dev'])}"
)

print(
    f"Test:  {len(reference_dataset['test'])}"
)


Loaded fixed reference dataset:
Train: 2963
Dev:   609
Test:  633


In [41]:
reference_split_by_text = {}

for split_name in ["train", "dev", "test"]:

    for text in reference_dataset[split_name]["text"]:

        if text in reference_split_by_text:
            raise ValueError(
                "Duplicate text found in reference dataset. "
                "Matching by text would be ambiguous.\n\n"
                f"Text:\n{text[:500]}"
            )

        reference_split_by_text[text] = split_name


print(
    f"\nMapped {len(reference_split_by_text)} "
    f"reference instances to their splits."
)


Mapped 4205 reference instances to their splits.


In [42]:
def labels_to_matrix(label_lists, valid_labels):

    label_to_index = {
        label: i
        for i, label in enumerate(valid_labels)
    }

    matrix = np.zeros(
        (len(label_lists), len(valid_labels)),
        dtype=np.float32,
    )

    for row_idx, labels in enumerate(label_lists):

        for label in labels:

            if label not in label_to_index:
                raise ValueError(
                    f"Unknown label '{label}'"
                )

            matrix[row_idx, label_to_index[label]] = 1.0

    return matrix

In [43]:
jsonl_files = sorted(
    INPUT_FOLDER.glob("*.jsonl")
)

print("\nJSONL files found:")

for filepath in jsonl_files:
    print(f"  {filepath.name}")


JSONL files found:
  Combined_hybrid_no_NA.jsonl
  Combined_ID_hybrid_no_NA.jsonl
  Combined_single_no_NA.jsonl
  Combined_SP_hybrid_no_NA.jsonl


In [44]:
for filepath in jsonl_files:

    # --------------------------------------------------------
    # Skip the reference file
    # --------------------------------------------------------

    if filepath.resolve() == REFERENCE_FILE.resolve():
        continue

    dataset_name = filepath.stem

    print("\n" + "=" * 70)
    print(f"Processing: {dataset_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Load this JSONL
    # --------------------------------------------------------

    texts, label_lists = load_jsonl(filepath)

    print(
        f"Number of instances: {len(texts)}"
    )

    # --------------------------------------------------------
    # Check number of instances
    # --------------------------------------------------------

    if len(texts) != len(reference_texts):

        raise ValueError(
            f"\n{dataset_name} does not contain the same "
            f"number of instances as the reference.\n\n"
            f"Reference: {len(reference_texts)}\n"
            f"Current:   {len(texts)}"
        )

    # --------------------------------------------------------
    # Check that every instance exists in reference
    # --------------------------------------------------------

    missing_instances = [
        text
        for text in texts
        if text not in reference_split_by_text
    ]

    if missing_instances:

        raise ValueError(
            f"\n{dataset_name} contains "
            f"{len(missing_instances)} instances "
            f"that are not present in the reference dataset."
        )

    # --------------------------------------------------------
    # Check that there are no missing reference instances
    # --------------------------------------------------------

    current_texts = set(texts)
    reference_texts_set = set(reference_texts)

    missing_from_current = (
        reference_texts_set - current_texts
    )

    if missing_from_current:

        raise ValueError(
            f"\n{dataset_name} is missing "
            f"{len(missing_from_current)} instances "
            f"from the reference dataset."
        )

    print("Instance check: PASSED")

    # --------------------------------------------------------
    # Determine labels used by this dataset
    # --------------------------------------------------------

    dataset_labels = sorted(
        set(
            label
            for labels in label_lists
            for label in labels
        )
    )

    print("\nLabels:")
    print(dataset_labels)

    # --------------------------------------------------------
    # Convert labels to matrix
    # --------------------------------------------------------

    y = labels_to_matrix(
        label_lists,
        all_valid_labels,
    )

    # --------------------------------------------------------
    # Build train/dev/test lists
    # --------------------------------------------------------

    split_texts = {
        "train": [],
        "dev": [],
        "test": [],
    }

    split_labels = {
        "train": [],
        "dev": [],
        "test": [],
    }

    # --------------------------------------------------------
    # Assign each instance to the SAME split as reference
    # --------------------------------------------------------

    for text, label_vector in zip(texts, y):

        split_name = reference_split_by_text[text]

        split_texts[split_name].append(text)
        split_labels[split_name].append(
            label_vector
        )

    # --------------------------------------------------------
    # Create Hugging Face datasets
    # --------------------------------------------------------

    train_dataset = Dataset.from_dict(
        {
            "text": split_texts["train"],
            "labels": np.array(
                split_labels["train"],
                dtype=np.float32,
            ).tolist(),
        }
    )

    dev_dataset = Dataset.from_dict(
        {
            "text": split_texts["dev"],
            "labels": np.array(
                split_labels["dev"],
                dtype=np.float32,
            ).tolist(),
        }
    )

    test_dataset = Dataset.from_dict(
        {
            "text": split_texts["test"],
            "labels": np.array(
                split_labels["test"],
                dtype=np.float32,
            ).tolist(),
        }
    )

    # --------------------------------------------------------
    # Shuffle each split using the SAME seed
    # --------------------------------------------------------

    train_dataset = train_dataset.shuffle(
        seed=SEED
    )

    dev_dataset = dev_dataset.shuffle(
        seed=SEED
    )

    test_dataset = test_dataset.shuffle(
        seed=SEED
    )

    # --------------------------------------------------------
    # DatasetDict
    # --------------------------------------------------------

    dataset_dict = DatasetDict(
        {
            "train": train_dataset,
            "dev": dev_dataset,
            "test": test_dataset,
        }
    )

    # --------------------------------------------------------
    # Print split sizes
    # --------------------------------------------------------

    print("\nSplit sizes:")

    print(
        f"Train: {len(dataset_dict['train'])}"
    )

    print(
        f"Dev:   {len(dataset_dict['dev'])}"
    )

    print(
        f"Test:  {len(dataset_dict['test'])}"
    )

    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    output_dir = OUTPUT_ROOT / dataset_name

    if output_dir.exists():

        raise FileExistsError(
            f"\nOutput already exists:\n"
            f"{output_dir}\n\n"
            f"Delete it manually if you want to "
            f"recreate it."
        )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # Save DatasetDict
    # --------------------------------------------------------

    dataset_dict.save_to_disk(
        str(output_dir)
    )

    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    metadata = {
        "dataset_name": dataset_name,
        "input_file": str(filepath),
        "reference_file": str(REFERENCE_FILE),
        "reference_dataset": REFERENCE_FILE.stem,
        "split_source": "fixed reference dataset",
        "seed": SEED,
        "num_examples": len(texts),
        "num_labels": len(all_valid_labels),
        "labels": all_valid_labels,
        "labels_present_in_source": dataset_labels,
        "same_instances_as_reference": True,
    }

    metadata_file = (
        output_dir / "split_metadata.json"
    )

    with open(
        metadata_file,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"\nSaved to:\n{output_dir}"
    )


Processing: Combined_hybrid_no_NA
Number of instances: 4205
Instance check: PASSED

Labels:
['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'ra', 're', 'rs', 'rv', 'sr']

Split sizes:
Train: 2963
Dev:   609
Test:  633


Saving the dataset (1/1 shards): 100%|██████████| 633/633 [00:00<00:00, 55575.21 examples/s]


Saved to:
C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_hybrid_no_NA

Processing: Combined_ID_hybrid_no_NA


Number of instances: 4205
Instance check: PASSED

Labels:
['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'ra', 're', 'rs', 'rv', 'sr']

Split sizes:
Train: 2963
Dev:   609
Test:  633


Saving the dataset (1/1 shards): 100%|██████████| 633/633 [00:00<00:00, 36685.89 examples/s]


Saved to:
C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_ID_hybrid_no_NA

Processing: Combined_SP_hybrid_no_NA


Number of instances: 4205
Instance check: PASSED

Labels:
['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'ra', 're', 'rs', 'rv', 'sr']

Split sizes:
Train: 2963
Dev:   609
Test:  633


Saving the dataset (1/1 shards): 100%|██████████| 633/633 [00:00<00:00, 23762.17 examples/s]


Saved to:
C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_SP_hybrid_no_NA


# Verification

In [46]:
from pathlib import Path
from datasets import load_from_disk

DATASET_ROOT = Path(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA"
)

In [47]:
dataset_dirs = sorted(
    [
        p for p in DATASET_ROOT.iterdir()
        if p.is_dir() and (p / "dataset_dict.json").exists()
    ]
)

print("Datasets found:")
for path in dataset_dirs:
    print(f"  {path.name}")

print(f"\nTotal datasets: {len(dataset_dirs)}")

# %%
# Load all datasets
datasets = {}

for path in dataset_dirs:
    datasets[path.name] = load_from_disk(str(path))

Datasets found:
  Combined_hybrid_no_NA
  Combined_ID_hybrid_no_NA
  Combined_single_no_NA
  Combined_SP_hybrid_no_NA

Total datasets: 4


In [48]:
def verify_instance(split_name, index):
    
    print("\n" + "=" * 100)
    print(f"SPLIT: {split_name.upper()} | INSTANCE INDEX: {index}")
    print("=" * 100)

    reference_text = None

    for dataset_name, dataset in datasets.items():

        example = dataset[split_name][index]

        text = example["text"]
        labels = example["labels"]

        print("\n" + "-" * 100)
        print(f"DATASET: {dataset_name}")
        print("-" * 100)

        print(f"Text:\n{text}")
        print(f"\nLabels:\n{labels}")

        # ----------------------------------------------------
        # Check text against reference
        # ----------------------------------------------------

        if reference_text is None:
            reference_text = text

        else:
            if text == reference_text:
                print("\n✓ Text matches reference")
            else:
                print("\n✗ WARNING: Text DOES NOT match reference")

In [49]:
for split in ["train", "dev", "test"]:
    verify_instance(
        split_name=split,
        index=4,
    )


SPLIT: TRAIN | INSTANCE INDEX: 4

----------------------------------------------------------------------------------------------------
DATASET: Combined_hybrid_no_NA
----------------------------------------------------------------------------------------------------
Text:
ورزش میلان میلان در زمینه های ورزشی به ویژه فوتبال بسیار فعال است. دو باشگاه آ.ث. میلان (A.C.Milan) و اینتر میلان (Inter Milan) از باشگاه های معروف اروپا هستند. تیم ...
ادامهمکانهای دیدنی میلان میدان اصلی شهر ، پیاتزا دل دومو (Piazza del Duomo) نام دارد که به دومو یا کلیسای جامع ختم می شود. دومو ساختمانی از مرمر سفید با سبک ...
ادامهدانشگاههای میلان دانشگاه پلی تکنیک ایتالیا ، بزرگترین دانشگاه فنی در ایتالیا است که در بیست و ششم نوامبر 1863 افتتاح شد. در میلان دانشگاهها ، مدارس عالی و مراکز ...
ادامهاقتصاد شهر میلان میلان مرکز اصلی دانش وهنر ایتالیا و از جمله مراکز مهم اقتصادی ، تجاری و صنعتی و مرکز مد و طراحی لباس در ایتالیاست. بنا به برخی گزارش ها ، میلان ...
ادامهفرودگاه های میلان میلان دارای ساختمانهای بلند مسکون

In [50]:
reference_name = "Combined_single_no_NA"

reference_dataset = datasets[reference_name]

for split in ["train", "dev", "test"]:

    reference_texts = reference_dataset[split]["text"]

    print(
        f"\nChecking {split}: "
        f"{len(reference_texts)} instances"
    )

    for dataset_name, dataset in datasets.items():

        current_texts = dataset[split]["text"]

        if current_texts == reference_texts:
            print(
                f"  ✓ {dataset_name}: identical"
            )
        else:
            print(
                f"  ✗ {dataset_name}: DIFFERENT"
            )

print("\nVerification complete.")


Checking train: 2963 instances
  ✓ Combined_hybrid_no_NA: identical
  ✓ Combined_ID_hybrid_no_NA: identical
  ✓ Combined_single_no_NA: identical
  ✓ Combined_SP_hybrid_no_NA: identical

Checking dev: 609 instances
  ✓ Combined_hybrid_no_NA: identical
  ✓ Combined_ID_hybrid_no_NA: identical
  ✓ Combined_single_no_NA: identical
  ✓ Combined_SP_hybrid_no_NA: identical

Checking test: 633 instances
  ✓ Combined_hybrid_no_NA: identical
  ✓ Combined_ID_hybrid_no_NA: identical
  ✓ Combined_single_no_NA: identical
  ✓ Combined_SP_hybrid_no_NA: identical

Verification complete.
